In [1]:
import numpy as np
import pandas as pd
import datetime
from pathlib import Path

In [2]:
root = Path("..").resolve()
raw_path = root / "data" / "raw"
train_path = raw_path / "2022-24"
test_path = raw_path / "2025"
# train vix from train path, test vix from test path
train_vix = train_path / "vix.csv"
test_vix = test_path / "vix.csv"
dates_path = raw_path / "dates"

In [3]:
df = pd.read_csv(train_path/ "RELIANCE.csv")
df.tail(10)

,20220103,09:07:35,2365.00,1,0,2364.0,1425,2365.0,4478
14276955,20241231,15:29:52,1215.95,302,0,1215.90,15,1216.00,7066
14276956,20241231,15:29:53,1216.00,489,0,1215.90,15,1216.00,6577
14276957,20241231,15:29:54,1215.65,38,0,1215.65,46,1216.00,6658
14276958,20241231,15:29:55,1216.00,1547,0,1215.80,1000,1216.00,6531
14276959,20241231,15:29:56,1215.90,285,0,1215.80,1005,1215.90,84
14276960,20241231,15:29:58,1216.00,6548,0,1216.00,11,1216.05,3
14276961,20241231,15:29:59,1216.05,10,0,1216.00,39,1216.05,1
14276962,20241231,15:30:00,1216.40,500,0,1216.00,65,1216.40,20
14276963,20241231,15:31:11,1215.45,1,0,0.00,0,0.00,0
14276964,20241231,16:00:00,1215.45,1918,0,1215.45,79,0.00,0


In [4]:
symbols = ["BAJFINANCE", "RELIANCE"]
months = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]
train_years = ["22","23","24"]

columns_fut = ["day","time","fut_ltp","fut_ltq","oi_fut","fut_bidprice","fut_bidvol","fut_askprice","fut_askvol"]
#sample column : 20220128,09:17:34,6994.60,125,500,6986.30,125,7030.70,125

columns_eq = ["day","time","ltp","ltq","oi","eq_bidprice","eq_bidvol","eq_askprice","eq_askvol"]
#sample column : 20220103 09:15:00 2364.00 	57496	0	2363.05	 235	 2364.00 	199

make individual dataframes corresponding to (dte x time bucket). 
dte calculations : based on dates/trading_calendar.csv and dates/expiry_dates.csv

read sequentially from each csv, find the pattern {symbol}{yy}{mmm}FUT.csv :
df = pd.read_csv

In [5]:
# create dataframes corresponding to dte x time bucket : expected 90 x 7 = 630 approximate
trading_cal = pd.read_csv(dates_path / "trading_calendar.csv", parse_dates=["date"])
expiry_dates = pd.read_csv(dates_path / "expiry_dates.csv", parse_dates=["expiry_date"])
trading_days_set = set(trading_cal["date"])

#make 1 hour buckets. the time corresponds to bucket starts
# 915-10,10-11,11-12,12-13,13-14,14-15,15-1530
buckets = ["9:15:00","10:00:00","11:00:00","12:00:00","13:00:00","14:00:00","15:00:00"]

In [6]:
# sorted trading calendar and expiry list for lookup
trading_days_sorted = sorted(trading_days_set)
expiry_list = sorted(expiry_dates["expiry_date"])


def find_first_gte(arr, target):
    """Binary search: return index of the first element >= target."""
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo


def find_exact(arr, target):
    """Binary search for exact match. Returns index or None."""
    idx = find_first_gte(arr, target)
    if idx < len(arr) and arr[idx] == target:
        return idx
    return None


def get_dte(day_str):
    """Calculate DTE = trading days from cur to the nearest expiry >= cur.

    1. Parse day to Timestamp.
    2. Binary search expiry_list for nearest expiry >= cur.
    3. Binary search trading_days_sorted for index of cur.
    4. Binary search trading_days_sorted for index of expiry.
       DTE = index(expiry) - index(cur).
       (cur is on expiry day itself -> DTE = 0.)
    """
    cur = pd.Timestamp(str(day_str))

    # nearest expiry >= cur
    ei = find_first_gte(expiry_list, cur)
    if ei >= len(expiry_list):
        return np.nan
    expiry = expiry_list[ei]

    # binary search trading calendar for both dates
    idx_cur = find_exact(trading_days_sorted, cur)
    idx_exp = find_exact(trading_days_sorted, expiry)

    if idx_cur is None or idx_exp is None:
        return np.nan

    return idx_exp - idx_cur


def get_bucket(time_str):
    """Assign a bucket index (0, 1, 2, ...) based on bucket start times.

    Bucket i covers [buckets[i], buckets[i+1]).
    The last bucket covers [15:00:00, end of day).
    """
    t = pd.Timedelta(time_str)
    bucket_td = [pd.Timedelta(b) for b in buckets]
    for i in range(len(bucket_td) - 1, -1, -1):
        if t >= bucket_td[i]:
            return i
    return 0

In [7]:
# vix function
# read INDIAVIX.csv from train path (same headerless format: day, time, ltp, ...)
columns_vix = ["day", "time", "vix","vol_vix","oi_vix"]
#sample column :20250101,09:15:05,14.39,0,0


df_vix = pd.read_csv(train_path / "INDIAVIX.csv", header=None, names=columns_vix)

vix_thresh = 20  # set vix threshold


def get_high_vix_days(df_vix, threshold=vix_thresh):
    """Return a set of days where VIX crossed the threshold at any tick.

    For each unique day in the VIX data, check all ticks from that day.
    If the threshold is crossed even once, the whole day is flagged.
    """
    high_vix_days = set()
    for day, day_ticks in df_vix.groupby("day"):
        if (day_ticks["vix"] >= threshold).any():
            high_vix_days.add(day)
    return high_vix_days


high_vix_days = get_high_vix_days(df_vix)
print(f"{len(high_vix_days)} days excluded out of {df_vix['day'].nunique()} total (VIX >= {vix_thresh})")

144 days excluded out of 740 total (VIX >= 20)


In [8]:
# processed data output directory
processed_path = root / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

# clear old processed CSVs to avoid duplicates from re-runs
for old_csv in processed_path.glob("*.csv"):
    old_csv.unlink()


def store_group(df_group, symbol, dte, bucket):
    """Append a DataFrame chunk to the appropriate CSV.

    File naming: {symbol}_{dte}_{bucket}.csv
    Writes header on first write; appends without header thereafter.
    """
    csv_path = processed_path / f"{symbol}_{int(dte)}_{int(bucket)}.csv"
    header = not csv_path.exists()
    df_group.to_csv(csv_path, mode="a", header=header, index=False)

In [9]:
for sym in symbols:
    for yr in train_years:
        for mon in months:
            # locate files
            eq_file = train_path / f"{sym}.csv"
            fut_file = train_path / f"{sym}{yr}{mon}FUT.csv"
            if not fut_file.exists():
                continue

            # read tick data (headerless CSVs)
            df_eq = pd.read_csv(eq_file, header=None, names=columns_eq)
            df_fut = pd.read_csv(fut_file, header=None, names=columns_fut)

            # create timestamp by merging day and time
            df_eq["timestamp"] = pd.to_datetime(df_eq["day"].astype(str) + " " + df_eq["time"])
            df_fut["timestamp"] = pd.to_datetime(df_fut["day"].astype(str) + " " + df_fut["time"])

            # sort by timestamp (required for merge_asof)
            #df_eq = df_eq.sort_values("timestamp")
            #df_fut = df_fut.sort_values("timestamp")

            # merge_asof: for each futures tick, grab the nearest prior equity tick
            df = pd.merge_asof(df_fut, df_eq, on="timestamp", direction="backward", suffixes=("_fut", "_eq"))

            # filter out high VIX days
            df = df[~df["day_fut"].isin(high_vix_days)]

            # compute DTE via binary search on trading calendar and expiry dates
            df["dte"] = df["day_fut"].apply(get_dte)

            # compute bucket per row
            df["bucket"] = df["time_fut"].apply(get_bucket)

            # tag each tick with its source contract month (e.g. "JAN")
            df["contract"] = mon

            # drop rows where DTE could not be computed
            df = df.dropna(subset=["dte"])

            # group by (dte, bucket) and append each group to its CSV
            for (dte, bucket), grp in df.groupby(["dte", "bucket"]):
                store_group(grp, sym, dte, bucket)

            n_csvs = df.groupby(["dte", "bucket"]).ngroups
            print(f"{sym}{yr}{mon}FUT — {len(df)} rows saved across {n_csvs} CSVs")

BAJFINANCE22JANFUT — 103410 rows saved across 175 CSVs
BAJFINANCE22FEBFUT — 38181 rows saved across 165 CSVs
BAJFINANCE22MARFUT — 3024 rows saved across 100 CSVs
BAJFINANCE22APRFUT — 27496 rows saved across 63 CSVs
BAJFINANCE22MAYFUT — 9431 rows saved across 70 CSVs
BAJFINANCE22JUNFUT — 9647 rows saved across 67 CSVs
BAJFINANCE22JULFUT — 67126 rows saved across 117 CSVs
BAJFINANCE22AUGFUT — 96136 rows saved across 126 CSVs
BAJFINANCE22SEPFUT — 63070 rows saved across 140 CSVs
BAJFINANCE22OCTFUT — 42797 rows saved across 140 CSVs
BAJFINANCE22NOVFUT — 102887 rows saved across 141 CSVs
BAJFINANCE22DECFUT — 128261 rows saved across 175 CSVs
BAJFINANCE23JANFUT — 37953 rows saved across 175 CSVs
BAJFINANCE23FEBFUT — 102123 rows saved across 168 CSVs
BAJFINANCE23MARFUT — 32213 rows saved across 140 CSVs
BAJFINANCE23APRFUT — 80388 rows saved across 140 CSVs
BAJFINANCE23MAYFUT — 113409 rows saved across 133 CSVs
BAJFINANCE23JUNFUT — 108858 rows saved across 168 CSVs
BAJFINANCE23JULFUT — 109562 

Building distributions
Have some waiting period : wait for 3 months and build up distributions

there will be one distribution for each (dte, bucket) value

no lookahead bias, dont include current point while building its distribution

for row i (in the dataframe) we look at all points from index 0 to i-1.

append the distribution parameters to the corresponding row in the correct table

In [10]:
from dateutil.relativedelta import relativedelta

warmup_months = 3

# prepare VIX with timestamps for merge_asof
df_vix["timestamp"] = pd.to_datetime(df_vix["day"].astype(str) + " " + df_vix["time"])
df_vix_sorted = df_vix[["timestamp", "vix"]].sort_values("timestamp").reset_index(drop=True)

# sort by (symbol, dte, bucket) — filename is {symbol}_{dte}_{bucket}.csv
csv_files = sorted(
    processed_path.glob("*_*_*.csv"),
    key=lambda p: (p.stem.rsplit("_", 2)[0], int(p.stem.rsplit("_", 2)[1]), int(p.stem.rsplit("_", 2)[2]))
)
print(f"Processing {len(csv_files)} (symbol, dte, bucket) CSVs...")

for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # drop columns from partial prior runs to avoid duplicates
    df = df.drop(columns=["spread", "dist_mean", "dist_std", "dist_count", "vix", "vix_x", "vix_y"], errors="ignore")

    # drop 2021 rows and rows with missing equity data
    df = df[df["day_fut"] >= 20220101]
    df = df.dropna(subset=["ltp"])

    # re-apply VIX day filter (original run had broken VIX columns)
    df = df[~df["day_fut"].isin(high_vix_days)]

    if df.empty:
        csv_path.unlink()
        print(f"{csv_path.name}: deleted (empty after filtering)")
        continue

    
    # normalized basis spread (%)
    df["spread"] = ((df["fut_ltp"] - df["ltp"]) / df["ltp"]) * 100

    # expanding stats shifted by 1: row i gets stats from rows 0..i-1
    expanding = df["spread"].expanding()
    df["dist_mean"] = expanding.mean().shift(1)
    df["dist_std"] = expanding.std().shift(1)
    df["dist_count"] = expanding.count().shift(1)

    # mask the 3-month warm-up period
    cutoff = df["timestamp"].iloc[0] + relativedelta(months=warmup_months)
    warmup_mask = df["timestamp"] < cutoff
    df.loc[warmup_mask, ["dist_mean", "dist_std", "dist_count"]] = np.nan

    # write back to the same processed CSV
    df.to_csv(csv_path, index=False)

    print(f"{csv_path.name}: {len(df)} rows, {(~warmup_mask).sum()} with distributions")

Processing 350 (symbol, dte, bucket) CSVs...
BAJFINANCE_0_0.csv: 52026 rows, 45506 with distributions
BAJFINANCE_0_1.csv: 53011 rows, 47512 with distributions
BAJFINANCE_0_2.csv: 45418 rows, 40916 with distributions
BAJFINANCE_0_3.csv: 43931 rows, 39679 with distributions
BAJFINANCE_0_4.csv: 46896 rows, 42988 with distributions
BAJFINANCE_0_5.csv: 53404 rows, 49559 with distributions
BAJFINANCE_0_6.csv: 35717 rows, 32614 with distributions
BAJFINANCE_1_0.csv: 47885 rows, 44545 with distributions
BAJFINANCE_1_1.csv: 53719 rows, 51010 with distributions
BAJFINANCE_1_2.csv: 48835 rows, 45594 with distributions
BAJFINANCE_1_3.csv: 44388 rows, 41638 with distributions
BAJFINANCE_1_4.csv: 47170 rows, 44614 with distributions
BAJFINANCE_1_5.csv: 50268 rows, 47757 with distributions
BAJFINANCE_1_6.csv: 33058 rows, 29286 with distributions
BAJFINANCE_2_0.csv: 53443 rows, 46951 with distributions
BAJFINANCE_2_1.csv: 57293 rows, 51114 with distributions
BAJFINANCE_2_2.csv: 48317 rows, 42922 with 

Make features